# Divar Fatemi Decision Scanner v0.4.2

این نسخه **Laya Multilingual** را به‌عنوان موتور اصلی System‑1 اجرا می‌کند: mmBERT encoder + decision head، غیر-autoregressive، بدون text generation، با primitiveهای `choice / score / noul`.

Laya Multilingual/ParsBERT NLI فقط baseline پژوهشی هستند تا بتوانیم روی labelهای خودِ مسکن تهران مقایسهٔ واقعی انجام دهیم.

> خروجی «تقلب قطعی» نیست؛ هدف، اولویت‌بندی بازبینی و ساخت دیتاست انسانی برای fine-tune/calibration است.

In [ ]:
import os, sys, shutil, subprocess, pathlib, importlib

REPO = '/content/divar_scanner_repo'
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git', 'clone', 'https://github.com/prestigegitserp/divar_scanner.git', REPO], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[decision]'], check=True)

SRC = str(pathlib.Path(REPO, 'src').resolve())
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
for name in list(sys.modules):
    if name == 'divar_scanner' or name.startswith('divar_scanner.'):
        del sys.modules[name]
importlib.invalidate_caches()

import divar_scanner
import pandas as pd
import numpy as np
import sklearn
import torch

pkg = pathlib.Path(divar_scanner.__file__).resolve()
print('divar_scanner:', divar_scanner.__version__, pkg)
print('pandas:', pd.__version__, '| numpy:', np.__version__, '| sklearn:', sklearn.__version__)
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert str(pkg).replace('\\\\', '/').endswith('/src/divar_scanner/__init__.py'), f'WRONG PACKAGE ORIGIN: {pkg}'
print('✅ package/import self-check passed')


In [ ]:
subprocess.run([sys.executable, '-m', 'divar_scanner.doctor'], check=True)


## Acquisition: برای Colab از Kenar رسمی استفاده کن

`divar.ir` و `api.divar.ir` ممکن است از بعضی runtimeهای Colab اصلاً TCP-connect نشوند. مسیر پیش‌فرض این notebook، API رسمی **Kenar/Open Platform** روی `open-api.divar.ir` است.

- `kenar`: مسیر رسمی؛ نیازمند `KENAR_API_KEY` با permission `SEARCH_POST`.
- `snapshot`: هیچ crawl شبکه‌ای انجام نمی‌دهد؛ CSV/Parquet نرمال‌شده را upload می‌کنی و فقط تحلیل/Laya اجرا می‌شود.
- `web` / `api`: فقط برای محیط‌هایی که direct Divar connectivity دارند.

> `SEARCH_POST` طبق مستندات دیوار سهمیهٔ بسیار محدودی دارد و pagination ندارد؛ آن را برای snapshot استفاده کن، نه polling مداوم.


## تنظیمات مدل تصمیم‌گیر

- `laya-multilingual`: **پیش‌فرض و انتخاب اصلی**؛ System‑1 واقعی، RLCD، بدون generation، Apache‑2.0.
- `mdeberta-nli`: baseline چندزبانهٔ NLI با مجوز MIT.
- `parsbert-parsinlu`: baseline متخصص فارسی ParsiNLU با مجوز CC-BY-NC-SA-4.0.
- `persian-ensemble`: ensemble پژوهشی NLI برای مقایسه.

Laya base را accuracy نهایی فرض نکن؛ diagnostic و سپس label انسانی برای تخصصی‌سازی روی آگهی مسکن لازم است.

In [ ]:
import getpass, yaml
from pathlib import Path

ACQUISITION_MODE = 'kenar'  #@param ['kenar','snapshot','web','api']
MAX_LISTINGS = 100  #@param {type:"integer"}
RUN_DECISION_MODEL = True  #@param {type:"boolean"}
DECISION_BACKEND = 'laya-multilingual'  #@param ['laya-multilingual','mdeberta-nli','parsbert-parsinlu','mbert-parsinlu','persian-ensemble']
DECISION_TOP_K = 50  #@param {type:"integer"}
EXPLORATION_SAMPLE = 24  #@param {type:"integer"}
PERMUTATION_PASSES = 2  #@param {type:"integer"}
RUN_PERSIAN_DIAGNOSTIC = True  #@param {type:"boolean"}

cfg = yaml.safe_load(Path('config/fatemi.yaml').read_text(encoding='utf-8'))
cfg['crawl']['transport'] = ACQUISITION_MODE if ACQUISITION_MODE != 'snapshot' else 'kenar'
cfg['crawl']['max_listings'] = min(int(MAX_LISTINGS), 100) if ACQUISITION_MODE == 'kenar' else int(MAX_LISTINGS)
cfg['decision']['enabled'] = bool(RUN_DECISION_MODEL)
cfg['decision']['backend'] = DECISION_BACKEND
cfg['decision']['top_k'] = int(DECISION_TOP_K)
cfg['decision']['exploration_sample'] = int(EXPLORATION_SAMPLE)
cfg['decision']['permutation_passes'] = max(1, min(int(PERMUTATION_PASSES), 6))
cfg['decision']['strict'] = True

if ACQUISITION_MODE == 'kenar':
    key = getpass.getpass('KENAR_API_KEY with SEARCH_POST permission (hidden): ').strip()
    if not key:
        raise ValueError('KENAR_API_KEY لازم است. اگر کلید نداری ACQUISITION_MODE=snapshot را انتخاب کن.')
    os.environ['KENAR_API_KEY'] = key

runtime_cfg = Path('/content/fatemi_colab.yaml')
runtime_cfg.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('acquisition:', ACQUISITION_MODE)
print('runtime config:', runtime_cfg)
subprocess.run([sys.executable, '-m', 'divar_scanner.doctor', '--network'], check=True)


In [ ]:
if RUN_DECISION_MODEL and RUN_PERSIAN_DIAGNOSTIC:
    from divar_scanner.decision import create_decision_engine
    from divar_scanner.decision_benchmark import run_diagnostic
    from IPython.display import display

    engine, specs, temperatures = create_decision_engine(
        cfg['decision'], backend_override=DECISION_BACKEND
    )
    print('decision backend:', engine.backend_name)
    print('decision model:', engine.model_name)
    print('model metadata:')
    for spec in specs:
        print(' -', spec.backend, '=>', spec.model_name, '|', spec.license, '| engine:', spec.engine)
    diagnostic = run_diagnostic(engine)
    print(diagnostic['warning'])
    print('diagnostic score:', diagnostic['correct'], '/', diagnostic['total'])
    display(pd.DataFrame(diagnostic['rows']))

    del engine
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### تفسیر خروجی v0.4\nسه کانال مستقل داریم: `data_problem_score` برای خرابی/ناسازگاری داده، `market_outlier_score` برای پرت بازار، و `misleading_risk_score` برای شواهد گمراه‌کنندگی/bait. هیچ‌کدام به‌تنهایی اثبات تقلب نیستند.\n\nLaya خروجی آزاد تولید نمی‌کند. همهٔ primitiveها به گزینه‌های بسته و opaque تبدیل می‌شوند؛ دو ترتیب متفاوت گزینه‌ها اجرا و pool می‌شوند. `decision_order_stability` و `decision_coherence_score` پایین باعث `decision_abstain=True` می‌شوند.\n

In [ ]:
from divar_scanner.pipeline import run_pipeline

if ACQUISITION_MODE == 'snapshot':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('هیچ فایل CSV/Parquet آپلود نشد.')
    snapshot_path = next(iter(uploaded.keys()))
    if not snapshot_path.lower().endswith(('.csv', '.parquet')):
        raise ValueError('snapshot باید CSV یا Parquet نرمال‌شده باشد.')
    df, meta = run_pipeline('/content/fatemi_colab.yaml', crawl=False, input_path=snapshot_path)
else:
    df, meta = run_pipeline('/content/fatemi_colab.yaml', crawl=True)

print('COUNTS:', meta.get('counts'))
print('CRAWL:', meta.get('crawl', meta.get('crawl_transport')))
print('DECISION:', meta.get('decision'))
print('OUTPUTS:', meta.get('outputs'))


In [ ]:
from IPython.display import display

cols = [
    'review_priority_score', 'risk_band',
    'misleading_risk_score', 'data_problem_score', 'market_outlier_score',
    'uncertainty_score', 'title', 'neighborhood', 'area_m2', 'rooms', 'contract_style',
    'deposit_toman', 'rent_monthly_toman', 'price_model_ratio',
    'duplicate_cluster_size', 'duplicate_bait_score',
    'decision_evaluated', 'decision_disposition', 'decision_integrity_class',
    'decision_duplicate_pattern', 'decision_market_status',
    'decision_bait_probability', 'decision_data_error_probability',
    'decision_manual_review_probability', 'decision_consistency_score',
    'decision_effective_confidence', 'decision_order_stability',
    'decision_coherence_score', 'decision_abstain', 'decision_abstain_reasons',
    'flag_reasons', 'url'
]
display(df[[c for c in cols if c in df.columns]].head(80))


In [ ]:
from pathlib import Path
from IPython.display import IFrame, display

report = meta.get('outputs', {}).get('html_report')
if report and Path(report).exists():
    display(IFrame(src=report, width='100%', height=900))


In [ ]:
# Optional downloads from Colab
from google.colab import files
for label, path in meta.get('outputs', {}).items():
    print(label, '=>', path)
# Example: files.download(meta['outputs']['review_csv'])


## منطق نسخهٔ 0.3

- contract-aware robust peer statistics و حساسیت 25/30/35 برای معادل‌سازی رهن/اجاره
- Isolation Forest + LOF + OOF price model بدون leakage
- duplicate graph برای copy/pasteهای متناقض
- evidence firewall: سؤال bait قیمت پرتِ تک‌آگهی را نمی‌بیند
- **Laya System‑1**: `choice / score / noul` در forward pass غیرمولد
- exploration sample از آگهی‌های عادی برای کاهش selection bias در calibration
- review CSV با ستون‌های label انسانی
- temperature calibration روی دادهٔ خودِ فاطمی
- NLI backends فقط baseline/ablation هستند، نه موتور اصلی